# 🌟 Northstar Services — Support Assistant (v1)

**Session 3 Task:** Upgrading the Northstar Triage Assistant with **LangChain**.
- Modular chat model initialization via `init_chat_model` (`google_genai:gemini-3.1-flash-lite`)
- Structured outputs via `model.with_structured_output(Triage, include_raw=True)`
- Externalized YAML prompt instructions (`prompt.yml`)
- Domain-specific tools decorated with `@tool` (`lookup_account`, `get_knowledge_base_article`)
- Autonomous agent drafting using LangChain's `create_agent`

## 1. Environment & LangChain Model Initialization
Using LangChain's unified `init_chat_model` interface to instantiate the Google GenAI provider.

In [12]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables
load_dotenv()

MODEL = "google_genai:gemini-3.1-flash-lite"
model = init_chat_model(MODEL)

## 2. Structured Triage Schema (Pydantic)
Enforce strict schema validation for classification metadata.

In [13]:
from typing import Literal
from pydantic import BaseModel, Field


class Triage(BaseModel):
    """Structured customer support inquiry classification contract."""
    category: Literal["billing", "technical", "account", "general"] = Field(
        description="The primary topic of the customer's message."
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How quickly this inquiry requires attention."
    )
    sentiment: Literal["negative", "neutral", "positive"] = Field(
        description="The customer's detected emotional tone."
    )
    need_human: bool = Field(
        description="True if this inquiry must be escalated to a human agent — e.g. refunds, billing disputes, cancellations, legal issues, or anger."
    )
    summary: str = Field(
        description="A concise, single-line summary of what the customer requires."
    )

## 3. Externalized Prompts (YAML)
Load system prompts from `prompt.yml` to maintain clear separation between business logic and instructions.

In [14]:
import yaml
from pathlib import Path

# Resolve prompt.yml path safely whether executed from root or task directory
prompt_file = Path("prompt.yml") if Path("prompt.yml").exists() else Path("Session_3/Task/prompt.yml")
_PROMPTS = yaml.safe_load(prompt_file.read_text(encoding="utf-8"))

TRIAGE_SYSTEM = _PROMPTS["triage_system"]
REPLY_SYSTEM = _PROMPTS["reply_system"]

print("Prompts loaded successfully!")

Prompts loaded successfully!


## 4. Token & Cost Accounting Utility
Collect token consumption metadata from LangChain response payloads to estimate API usage costs.

In [15]:
IN_PRICE_PER_MTOK = 0.25     # $ per million input tokens
OUT_PRICE_PER_MTOK = 1.50    # $ per million output tokens
_totals = {"input": 0, "output": 0}

def track(usage_metadata: dict) -> None:
    """Accumulate input and output token counts from LangChain usage metadata."""
    if usage_metadata:
        _totals["input"] += usage_metadata.get("input_tokens", 0)
        _totals["output"] += usage_metadata.get("output_tokens", 0)

def usage_report() -> str:
    """Calculate and return human-readable token and cost summary."""
    cost = (_totals["input"] / 1e6 * IN_PRICE_PER_MTOK
            + _totals["output"] / 1e6 * OUT_PRICE_PER_MTOK)
    inp = _totals["input"]
    out = _totals["output"]
    return f"{inp} in + {out} out tokens = ~${cost:.6f}"

## 5. Structured Triage Classifier
Leverage LangChain's `model.with_structured_output(Triage, include_raw=True)` to extract both the validated Pydantic model and raw token metadata.

In [16]:
def triage(message: str) -> Triage:
    """Classify incoming customer message using structured output binding."""
    triage_model = model.with_structured_output(Triage, include_raw=True)
    response = triage_model.invoke([
        {"role": "system", "content": TRIAGE_SYSTEM},
        {"role": "user", "content": message}
    ])
    track(response["raw"].usage_metadata)
    return response["parsed"]

## 6. Domain Tools Definition (`@tool`)
Define tools for the agent to query simulated account billing status and access knowledge base documentation.

In [17]:
from langchain.tools import tool

@tool
def lookup_account(customer_query: str) -> str:
    """Look up customer account details and current billing status using inquiry context."""
    if "charged twice" in customer_query.lower():
        return "Account status: Active. Issue detected: System error caused duplicate billing on May 1st."
    return "Account status: Active. No billing issues."

@tool
def get_knowledge_base_article(topic: str) -> str:
    """Fetch official help articles for technical issues, upload limits, or account settings."""
    if "email" in topic.lower():
        return "Article: To change your email address, navigate to Settings > Profile > Update Email."
    elif "crash" in topic.lower():
        return "Article: If the app crashes on upload, ensure the file is under 50MB and clear the app cache."
    return "No specific article found."

northstar_tools = [lookup_account, get_knowledge_base_article]

## 7. Agent Creation & Drafting Pipeline
Construct an autonomous support agent via `create_agent` that uses `northstar_tools` to verify facts before composing responses.

In [18]:
from langchain.agents import create_agent

# Create autonomous drafting agent
agent = create_agent(
    model=MODEL,
    tools=northstar_tools,
    system_prompt=REPLY_SYSTEM
)

def draft_agent_reply(message: str, t: Triage) -> str:
    """Use the LangChain agent with domain tools to draft a grounded customer reply."""
    user_payload = f"Customer message:\n{message}\n\nTriage context: {t.model_dump()}"
    
    result = agent.invoke({
        "messages": [{"role": "user", "content": user_payload}]
    })
    
    final_message = result["messages"][-1]
    if hasattr(final_message, "usage_metadata") and final_message.usage_metadata:
        track(final_message.usage_metadata)
        
    return final_message.text

## 8. Test Evaluation on Diverse Customer Scenarios
Run the complete pipeline against 6 customer inquiries spanning billing disputes, account updates, application crashes, and escalation queries.

In [19]:
def main() -> None:
    """Execute evaluation across sample support inquiries and report token usage."""
    samples = [
        "I've been charged twice for May and nobody has replied. This is ridiculous.",
        "How do I change the email address on my account?",
        "Your app keeps crashing when I upload a file.",
        "Who are you?",
        "What can you do for me",
        "Can I directly contact with the human support",
    ]
    for s in samples:
        t = triage(s)
        print("-" * 70)
        print("MESSAGE :", s)
        print("TRIAGE  :", t.model_dump())
        print("ESCALATE:", "yes -> human" if t.need_human else "no")
        print("DRAFT   :", draft_agent_reply(s, t))

    print("=" * 70)
    print("TOKENS  :", usage_report())

main()

----------------------------------------------------------------------
MESSAGE : I've been charged twice for May and nobody has replied. This is ridiculous.
TRIAGE  : {'category': 'billing', 'urgency': 'high', 'sentiment': 'negative', 'need_human': True, 'summary': 'Customer is reporting a double charge for May and expressing frustration over lack of support response.'}
ESCALATE: yes -> human
DRAFT   : I am very sorry for the frustration caused by this billing issue and the delay in our response. A specialized teammate is currently reviewing your account details to investigate the duplicate charge and will follow up with you shortly to resolve this.
----------------------------------------------------------------------
MESSAGE : How do I change the email address on my account?
TRIAGE  : {'category': 'account', 'urgency': 'low', 'sentiment': 'neutral', 'need_human': False, 'summary': 'Customer is asking for instructions on how to update the email address associated with their account.'}